In [32]:
import os
import numpy as np
import torch
from torchvision.transforms import transforms
from stormer.data.iterative_dataset import ERA5MultiLeadtimeDataset
from stormer.models.iterative_module import GlobalForecastIterativeModule
from stormer.models.hub.stormer import Stormer
from stormer.data.multi_step_datamodule import collate_fn_val
from torch.utils.data import DataLoader, Subset
from stormer.utils.metrics import lat_weighted_rmse
from datetime import datetime, timedelta

In [33]:

variables = [
    "2m_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "mean_sea_level_pressure",
    "geopotential_50",
    "geopotential_100",
    "geopotential_150",
    "geopotential_200",
    "geopotential_250",
    "geopotential_300",
    "geopotential_400",
    "geopotential_500",
    "geopotential_600",
    "geopotential_700",
    "geopotential_850",
    "geopotential_925",
    "geopotential_1000",
    "u_component_of_wind_50",
    "u_component_of_wind_100",
    "u_component_of_wind_150",
    "u_component_of_wind_200",
    "u_component_of_wind_250",
    "u_component_of_wind_300",
    "u_component_of_wind_400",
    "u_component_of_wind_500",
    "u_component_of_wind_600",
    "u_component_of_wind_700",
    "u_component_of_wind_850",
    "u_component_of_wind_925",
    "u_component_of_wind_1000",
    "v_component_of_wind_50",
    "v_component_of_wind_100",
    "v_component_of_wind_150",
    "v_component_of_wind_200",
    "v_component_of_wind_250",
    "v_component_of_wind_300",
    "v_component_of_wind_400",
    "v_component_of_wind_500",
    "v_component_of_wind_600",
    "v_component_of_wind_700",
    "v_component_of_wind_850",
    "v_component_of_wind_925",
    "v_component_of_wind_1000",
    "temperature_50",
    "temperature_100",
    "temperature_150",
    "temperature_200",
    "temperature_250",
    "temperature_300",
    "temperature_400",
    "temperature_500",
    "temperature_600",
    "temperature_700",
    "temperature_850",
    "temperature_925",
    "temperature_1000",
    "specific_humidity_50",
    "specific_humidity_100",
    "specific_humidity_150",
    "specific_humidity_200",
    "specific_humidity_250",
    "specific_humidity_300",
    "specific_humidity_400",
    "specific_humidity_500",
    "specific_humidity_600",
    "specific_humidity_700",
    "specific_humidity_850",
    "specific_humidity_925",
    "specific_humidity_1000",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f'device: {device}', flush=True)
# load pretrained model
net = Stormer(
    in_img_size=[128, 256],
    variables=variables,
    patch_size=4,
    hidden_size=1024,
    depth=24,
    num_heads=16,
    mlp_ratio=4,
)
pretrained_path = r'C:\Users\maxsh\Stormer\stormer_1.40625_patch_size_4.ckpt'
model = GlobalForecastIterativeModule(net, pretrained_path=pretrained_path).to(device)
model.eval()
print(f'model loaded!', flush=True)


device: cuda


Loading pre-trained checkpoint from: C:\Users\maxsh\Stormer\stormer_1.40625_patch_size_4.ckpt
<All keys matched successfully>
model loaded!


In [34]:
model.net

Stormer(
  (embedding): WeatherEmbedding(
    (token_embeds): ModuleList(
      (0-68): 69 x PatchEmbed(
        (proj): Conv2d(1, 1024, kernel_size=(4, 4), stride=(4, 4))
        (norm): Identity()
      )
    )
    (channel_agg): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
    )
  )
  (embed_norm_layer): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  (t_embedder): TimestepEmbedder(
    (mlp): Linear(in_features=1, out_features=1024, bias=True)
  )
  (blocks): ModuleList(
    (0-23): 24 x Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=False)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affi

In [35]:
from torchao import quantize_
from torchao.quantization.quant_api import Int8WeightOnlyConfig
import copy

In [36]:
model_w8 = copy.deepcopy(model)

In [37]:
def _safe_linear(m, fqn):
    return (
        isinstance(m, torch.nn.Linear)
        and m.weight.dim() == 2
        and m.weight.shape[1] > 1
    )

In [38]:
quantize_(model_w8, Int8WeightOnlyConfig(), filter_fn=_safe_linear)

In [39]:
# Verify what got quantized across ALL layer types (not just Linear).
# torchao's Int8WeightOnlyConfig only swaps nn.Linear weights, so Conv2d / LayerNorm /
# MultiheadAttention stay in their original dtype. We inspect every *leaf* module so we
# can see the whole model, then group by (type, weight-shape, status) to keep it readable.
from collections import Counter

def _is_quantized(w):
    return w is not None and "Quantized" in type(w).__name__

PRINT_EVERY_LAYER = False   # set True for the full, un-grouped per-layer dump

groups = Counter()
examples = {}
flat = []
for name, mod in model_w8.net.named_modules():
    if list(mod.children()):
        continue  # skip containers; only leaf layers
    w = getattr(mod, "weight", None)
    if w is None:
        shape, status = "-", "no weights"
    elif _is_quantized(w):
        shape, status = str(tuple(w.shape)), "INT8 (quantized)"
    else:
        shape, status = str(tuple(w.shape)), f"{w.dtype} (kept)"
    key = (type(mod).__name__, shape, status)
    groups[key] += 1
    examples.setdefault(key, name)
    flat.append((name, type(mod).__name__, shape, status))

print(f"{'module type':<24}{'weight shape':<16}{'status':<22}{'count':<7}example")
print("-" * 100)
for (mtype, shape, status), n in sorted(groups.items(), key=lambda x: (-x[1], x[0])):
    print(f"{mtype:<24}{shape:<16}{status:<22}{n:<7}{examples[(mtype, shape, status)]}")

if PRINT_EVERY_LAYER:
    print("\n--- every layer ---")
    for name, mtype, shape, status in flat:
        print(f"  {name:<44}{mtype:<22}{shape:<16}{status}")

# Footprint sanity check (int8 weight counts as 1 byte/elem, others use real dtype size).
def mb(m):
    total = 0
    for _, mod in m.named_modules():
        w = getattr(mod, "weight", None)
        if w is None or list(mod.children()):
            continue
        total += w.numel() * (1 if _is_quantized(w) else w.element_size())
    return total / 1e6

print(f"\nAll weighted layers: fp32={mb(model.net):.1f} MB  ->  w8={mb(model_w8.net):.1f} MB")

module type             weight shape    status                count  example
----------------------------------------------------------------------------------------------------
Dropout                 -               no weights            96     blocks.0.attn.attn_drop
Identity                -               no weights            94     embedding.token_embeds.0.norm
Conv2d                  (1024, 1, 4, 4) torch.float32 (kept)  69     embedding.token_embeds.0.proj
LayerNorm               -               no weights            48     blocks.0.norm1
SiLU                    -               no weights            25     blocks.0.adaLN_modulation.0
GELU                    -               no weights            24     blocks.0.mlp.act
Linear                  (1024, 1024)    INT8 (quantized)      24     blocks.0.attn.proj
Linear                  (1024, 4096)    INT8 (quantized)      24     blocks.0.mlp.fc2
Linear                  (3072, 1024)    INT8 (quantized)      24     blocks.0.attn.qkv
Lin